In [ ]:
import math
import matplotlib.pyplot as plt

import torch

# There is no model to train in this notebook, so the Hugging Face version only
# uses plain PyTorch tensor operations (identical to the PyTorch version)

plt.style.use("seaborn-v0_8-colorblind")

In [ ]:
# 16 frequencies spaced evenly on a log scale between 1 and 1000
frequencies = torch.exp(
    torch.linspace(
        math.log(1),
        math.log(1000),
        32 // 2,
    )
)

In [ ]:
embedding_list = []
for y in torch.arange(0, 1, 0.01):
    # PyTorch images are (batch, channels, height, width), so the noise
    # variance has shape (1, 1, 1, 1) and the embedding lives on dim 1
    x = y.reshape(1, 1, 1, 1)
    angular_speeds = 2.0 * math.pi * frequencies.reshape(1, -1, 1, 1)
    embeddings = torch.cat(
        [torch.sin(angular_speeds * x), torch.cos(angular_speeds * x)], dim=1
    )
    embedding_list.append(embeddings[0, :, 0, 0])
# (100 noise levels, 32 dims) -> (32, 100): one column per noise level
embedding_array = torch.stack(embedding_list).T

In [ ]:
plt.figure(figsize=(8, 6))
plt.imshow(
    embedding_array, cmap="coolwarm", interpolation="nearest", origin="lower"
)

plt.show()

In [ ]:
def positionalencoding1d(d_model, length):
    """
    :param d_model: dimension of the model
    :param length: length of positions
    :return: length*d_model position matrix
    """
    if d_model % 2 != 0:
        raise ValueError(
            "Cannot use sin/cos positional encoding with "
            "odd dim (got dim={:d})".format(d_model)
        )
    pe = torch.zeros(length, d_model)
    position = torch.arange(0, length).unsqueeze(1)  # column vector (length, 1)
    div_term = torch.exp(
        (torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model))
    )
    pe[:, : d_model // 2] = torch.sin(position * div_term)
    pe[:, d_model // 2 :] = torch.cos(position * div_term)

    return pe

In [ ]:
out = positionalencoding1d(32, 100).T
plt.imshow(out, cmap="coolwarm", interpolation="nearest", origin="lower")
plt.show()